## Analysis

In [1]:
import pandas as pd

from pathlib import Path
from plots import plot_column
from run_surrogate import (load_graph_data, get_device, load_trained_model, 
                           predict_next_step, save_prediction_csv)

# Case Parameters
SIM_NAME = "example-model"
CASE = "case0"


# Pathing
ROOT = Path().resolve().parent
BASE = ROOT / "sims" / SIM_NAME / "training_data" / "graph" / CASE
MODEL_PATH =  BASE / "model_graphsage.pt"

#### Load Surrogate Model

In [2]:
# Loading GNN model
data = load_graph_data(SIM_NAME, CASE)
device = get_device()
model = load_trained_model(MODEL_PATH, device=device)

#### Run Surrogate

In [3]:
STEP = 100

t = STEP #TODO  # choose a valid t (0 <= t < len(data["steps"]) - 1)
p_next, u_next, v_next = predict_next_step(model, t, data, device)
step_out = int(data["steps"][t+1]) if (t+1) < len(data["steps"]) else int(data["steps"][t])

RUN_RESULTS_PATH = BASE / f"{CASE}-predictions.csv"
save_prediction_csv(RUN_RESULTS_PATH, step_out, data["node_x"], data["node_y"], p_next, u_next, v_next)

Wrote predictions to /Volumes/connor/dev/projects/ns2d-surrogate/sims/example-model/training_data/graph/case0/case0-predictions.csv


#### Load Results

In [4]:
# Load simulation data
RESULTS_PATH = ROOT / "sims" / SIM_NAME / "training_data" / f"{CASE}-results.csv"

df_sim = pd.read_csv(RESULTS_PATH)
df_sim = df_sim[df_sim['step'] == STEP+1].copy()
df_sim.head(2)

,step,node_id,n_x,n_y,p,u,v,vn
1859208,101,0,-0.500000,0.00000,1.589118,-0.003050,0.00016,0.003054
1859209,101,1,-0.487464,0.11126,1.514442,-0.001938,0.00128,0.002323


In [5]:
# Load surrogate results

df_ml = pd.read_csv(RUN_RESULTS_PATH)
df_ml.head(2)

,step,node_id,n_x,n_y,p,u,v,vn
0,101,0,-0.500000,0.00000,1.570889,0.027360,-0.000734,0.027370
1,101,1,-0.487464,0.11126,1.510777,0.016912,0.001096,0.016948


In [6]:
df_ml.tail(2)

,step,node_id,n_x,n_y,p,u,v,vn
18406,101,18406,33.199966,-7.830775,0.974900,0.992705,0.014604,0.992812
18407,101,18407,33.954353,-7.830776,0.993751,0.999445,0.007510,0.999473


In [7]:
len(df_ml)

18408

In [8]:
# Quick checks
assert len(df_ml) != 0, "Missing surrogate data!"
assert len(df_sim) != 0, "Missing sim data!"
assert len(df_ml) == len(df_sim), f"Should be equal: {len(df_ml)}, {len(df_sim)}"

#### Process Results

In [15]:
# Calculate residuals at each node
#TODO: residual should be calculated as pct.

# Round to match extractor's 12-decimal uniqueness
df_sim[['n_x','n_y']] = df_sim[['n_x','n_y']].round(12)
df_ml[['n_x','n_y']]  = df_ml[['n_x','n_y']].round(12)

# Merge on node_id
pd.merge(
    df_sim, df_ml, on=['step','node_id'], how='inner', validate='one_to_one'
)

df_tmp = pd.merge(
    df_sim, df_ml, on=['step','node_id'], suffixes=('_true','_pred'), how='inner'
)

df_res = pd.DataFrame({
    'node_id': df_tmp['node_id'],
    'n_x': df_tmp['n_x_true'],
    'n_y': df_tmp['n_y_true'],
    'p': ((df_tmp['p_pred'] - df_tmp['p_true'])/df_tmp['p_true']).abs(),
    'u': ((df_tmp['u_pred'] - df_tmp['u_true'])/df_tmp['u_true']).abs(),
    'v': ((df_tmp['v_pred'] - df_tmp['v_true'])/df_tmp['v_true']).abs(),
    'vn': ((df_tmp['vn_pred'] - df_tmp['vn_true'])/df_tmp['vn_true']).abs(),
    'step': df_tmp['step']
})
df_res.describe()

,node_id,n_x,n_y,p,u,v,vn,step
count,18408.000000,18408.000000,18408.000000,18408.000000,1.840800e+04,18408.000000,1.840800e+04,18408.0
mean,9203.500000,10.569233,0.023535,0.006307,2.591197e-01,58.985228,8.418056e-02,101.0
std,5314.076213,10.912663,3.196556,0.010048,9.174572e+00,1138.751089,1.863630e+00,0.0
min,0.000000,-8.000000,-8.000000,0.000002,8.671065e-07,0.000013,4.188651e-07,101.0
25%,4601.750000,0.751685,-2.297408,0.001719,1.943171e-03,0.080918,1.947583e-03,101.0
50%,9203.500000,8.957236,0.037477,0.003453,3.477283e-03,1.156157,3.468175e-03,101.0
75%,13805.250000,19.353402,2.351493,0.006743,5.210798e-03,19.486311,5.195959e-03,101.0
max,18407.000000,35.000000,8.000000,0.204718,7.745728e+02,94507.832580,1.281718e+02,101.0


In [16]:
# Quick checks
assert len(df_ml) == len(df_sim) == len(df_res), f"Should be equal: {len(df_ml)}, {len(df_sim)}, {len(df_res)}"

#### Nodal Residual Analysis

In [17]:
# Plot
STEP = 101

fig1 = plot_column(df_res, col="p", case_name=0, step=101)
fig1.show()